In [14]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from bs4 import BeautifulSoup
from tqdm import tqdm
import re

from bs4 import XMLParsedAsHTMLWarning
import warnings

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

## **1. SCRAPER SETUP**

In [15]:
def bs4_extractor(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    return re.sub(r"\n\n+", "\n\n", soup.text).strip()

In [16]:
website_url = "https://pathway.com/developers/user-guide/introduction/pathway-overview/"
loader = RecursiveUrlLoader(
    website_url, extractor=bs4_extractor)

## **2. STREAMING & CHUNKING**

In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200)

In [18]:
all_chunks = []

# We use a loop to handle documents one-by-one (Streaming)
for doc in tqdm(loader.lazy_load(), bar_format="[INFO] {n_fmt} Pages Scraped."):

    # split_documents returns a LIST of chunks for this specific page
    chunks = text_splitter.split_documents([doc])

    # We use EXTEND to flatten those chunks into our main list
    all_chunks.extend(chunks)

print(f"[INFO] Created {len(all_chunks)} chunks.")

[INFO] 38 Pages Scraped.

[INFO] Created 994 chunks.


In [19]:
all_chunks[0]

Document(metadata={'source': 'https://pathway.com/developers/user-guide/introduction/pathway-overview/', 'content_type': 'text/html', 'title': 'Overview | Pathway', 'description': 'Quick introduction to Pathway operators.', 'language': 'en'}, page_content='Overview | PathwaydevelopersSearch⌘K62k Pathway Framework Developer Guide  Connectors  LLM Tooling  API Docs  Get Help IntroductionWelcomeInstallationOverviewStarting ExamplesCore ConceptsWhy PathwayLicensing GuideStreaming and Static ModesBatch Processing in PythonConnectTransformTemporal DataLLM toolingDeployMigratingAdvancedHelp And UpdatesGitHub0kQuick Overview of PathwayThis page is a summary of what you need to know about to start programming with Pathway.\nIf you want to learn more about the core concepts of Pathway, you can read the dedicated article.Import PathwayYou can quickly install Pathway with a simple pip command: pip install pathwayThen, you simply have to import Pathway as any other Python library:import pathway as 

## **3. EMBEDDINGS & VECTOR STORE**

In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")

# Pass the list of chunks directly - metadata is preserved automatically!
vector_store = InMemoryVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings
)

## **4. RETRIEVER**

In [21]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 10,
        "lambda_mult": 0.5
    }
)

In [22]:
llm = ChatOllama(model="deepseek-v3.1:671b-cloud", temperature=0.2)

## **5. PROMPT TEMPLATE**

In [23]:
# The chain expects {context} and {input}
prompt = ChatPromptTemplate.from_template("""
    You are a helpful and factual AI assistant.
    Use the following retrieved context to answer the user's question.
                                          
    If the answer is not found in the context, reply with:
    "I'm not sure based on the provided information."

    <context>
    {context}
    </context>

    Question: {input}
""")

## **6. THE CHAIN (The "Best" Method)**

In [24]:
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

## **7. Final Implementation**

In [ ]:
print("\n🚀 RAG System Ready! (Type '0' to exit)")
while True:
    query = input("\nYou: ")
    if query == "0":
        break

    # One call handles retrieval, prompt formatting, and LLM generation
    response = rag_chain.invoke({"input": query})

    print(f"\n🧠 AI: {response['answer']}")

    # Optional: show which URLs were used
    sources = {doc.metadata.get('source') for doc in response['context']}
    print(f"📍 Sources: {', '.join(sources)}")


🚀 RAG System Ready! (Type '0' to exit)

🧠 AI: Hello! How can I help you today?
📍 Sources: https://pathway.com/_nuxt/builds/meta/3f31ae12-13a7-4429-9063-501fcdfadf46.json, https://pathway.com/developers/user-guide/introduction/pathway-overview/_payload.json?3f31ae12-13a7-4429-9063-501fcdfadf46, https://pathway.com/developers/user-guide/data-transformation/indexes-in-pathway

🧠 AI: Based on the provided context, Pathway is a framework for data processing and transformation. It allows you to define processing pipelines modeled as a graph called a **dataflow**, where nodes represent operations on tables or connectors. It is used as a Python library that you install via `pip install pathway` and import with `import pathway as pw`.

The context also mentions that Pathway uses **schemas** to define the structure and data types (like `int`, `float`, `str`) of your data tables, ensuring organization and type safety.
📍 Sources: https://pathway.com/developers/user-guide/development/get-help, htt